# MODEL_CF — UserCF canonique

Ce notebook documente le modèle UserCF servi par NextWatch. Le protocole déterministe sépare les interactions en **65 % entraînement, 15 % validation et 20 % test** (`random_state=42`). Les hyperparamètres sont choisis uniquement sur la validation ; l’artefact est ensuite construit sur entraînement + validation, tandis que le test reste réservé à l’évaluation finale.

Les cas produit utilisent 1 à 3 films aimés, avec la définition canonique `rating > user_mean(reference)`.


## 1. Sources canoniques et imports


In [1]:
import json
import sys
from pathlib import Path

import joblib

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "scripts" / "evaluation_protocol.py").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nextwatch_recommend import recommend
from scripts.evaluation_protocol import (
    build_eval_cases,
    load_dataframe,
    protocol_metadata,
    split_dataframe,
)


In [2]:
params_path = ROOT / "results" / "best_hyperparams_cf_optuna.json"
raw_params = json.loads(params_path.read_text(encoding="utf-8"))
param_names = [
    "MIN_SUPPORT",
    "CF_N_USER",
    "DEMO_SEX_WEIGHT",
    "AGE_SCALE",
    "RATING_POWER",
]

print("Hyperparamètres UserCF :")
for name in param_names:
    print(f"  {name} = {raw_params[name]}")
print(f"Validation {raw_params['metric']} = {raw_params['validation_score']:.6f}")


Hyperparamètres UserCF :
  MIN_SUPPORT = 24
  CF_N_USER = 300
  DEMO_SEX_WEIGHT = 0.05
  AGE_SCALE = 60
  RATING_POWER = 2.0
Validation precision_at_5 = 0.069888


## 2. Split et cas d’évaluation canoniques


In [3]:
split = split_dataframe(load_dataframe())
validation_cases = build_eval_cases(split.train, split.validation)
test_cases = build_eval_cases(split.train_validation, split.test)

print(
    f"Split — train={len(split.train):,}, validation={len(split.validation):,}, "
    f"test={len(split.test):,}"
)
print(
    f"Cas — validation={len(validation_cases):,}, test={len(test_cases):,}"
)
print("Protocole :", protocol_metadata())


Split — train=64,993, validation=14,998, test=19,999
Cas — validation=890, test=921
Protocole : {'random_state': 42, 'train_frac': 0.65, 'validation_frac': 0.15, 'test_frac': 0.2, 'max_seeds': 3, 'liked_definition': 'rating > user_mean(reference)', 'seed_selection': 'deterministic_random_1_to_3'}


## 3. Artefact UserCF servi


In [4]:
artifact_path = ROOT / "artifacts" / "usercf.joblib"
artifact = joblib.load(artifact_path)

print("Paramètres de l’artefact :", artifact["params"])
print("Métadonnées de l’artefact :")
for key, value in artifact["meta"].items():
    print(f"  {key}: {value}")


Paramètres de l’artefact : {'MIN_SUPPORT': 24, 'CF_N_USER': 300, 'DEMO_SEX_WEIGHT': 0.05, 'AGE_SCALE': 60.0, 'RATING_POWER': 2.0}
Métadonnées de l’artefact :
  protocol: {'random_state': 42, 'train_frac': 0.65, 'validation_frac': 0.15, 'test_frac': 0.2, 'max_seeds': 3, 'liked_definition': 'rating > user_mean(reference)', 'seed_selection': 'deterministic_random_1_to_3'}
  dataset_sha256: e1bd23ec38015edb2730a20c11666d38ffb729d848cd34cae5196144c6c0406e
  runtime_versions: {'python': '3.14.4', 'numpy': '2.5.3', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'joblib': '1.6.0'}
  training_split: train+validation
  test_used: False
  tuning: {'validation_score': 0.06988764044943821, 'metric': 'precision_at_5', 'model': 'usercf', 'n_trials': 300, 'protocol': {'random_state': 42, 'train_frac': 0.65, 'validation_frac': 0.15, 'test_frac': 0.2, 'max_seeds': 3, 'liked_definition': 'rating > user_mean(reference)', 'seed_selection': 'deterministic_random_1_to_3'}}
  n_fans_films: 801
  n_likers: 943
  

## 4. Exemple d’inférence produit


In [5]:
recommendations = recommend(
    age=24,
    gender="M",
    liked_movies=[50, 174],
    model="cf",
    k=5,
)
recommendations


['Return of the Jedi (1983)',
 'Empire Strikes Back, The (1980)',
 'Princess Bride, The (1987)',
 'Silence of the Lambs, The (1991)',
 'Pulp Fiction (1994)']

## 5. Résultats finaux sur le test

Le **Hit Rate@5** mesure la part des cas ayant au moins un film aimé parmi les cinq recommandations.


In [6]:
evaluation_path = ROOT / "results" / "final_evaluation.json"
final_evaluation = json.loads(evaluation_path.read_text(encoding="utf-8"))
usercf_scores = final_evaluation["scores"]["user_user_cf"]

print(f"Cas test : {final_evaluation['n_cases']}")
print(f"P@5       = {usercf_scores['P@5']:.6f}")
print(f"P@10      = {usercf_scores['P@10']:.6f}")
print(f"Hit Rate@5 = {usercf_scores['HitRate@5']:.2%}")


Cas test : 921
P@5       = 0.093160
P@10      = 0.085342
Hit Rate@5 = 37.57%


## Conclusion

UserCF obtient P@5 = **0,0932**, P@10 = **0,0853** et Hit Rate@5 = **37,6 %** : plus d’un cas sur trois contient au moins un film réellement aimé dans le top 5. Le notebook reprend uniquement le chemin UserCF de production et ne lance aucun tuning ni export.


In [7]:
# Fin du notebook canonique UserCF.